## AI Resume Analyzer with resume classification and Job Matching

### Project Overview
The AI Resume Matching System is an intelligent recruitment support application that automatically analyzes resumes and compares them with job descriptions using Natural Language Processing (NLP) and Machine Learning techniques.
The system helps recruiters and candidates by:
- Extracting important skills from resumes
- Understanding job descriptions
- Calculating similarity scores
- Providing ATS-style matching results
- Recommending suitable jobs to candidates
  
This project simulates how modern Applicant Tracking Systems (ATS) work in real companies.

### Problem Statement
In traditional recruitment:
- Recruiters manually screen hundreds of resumes
- Candidates are rejected because resumes are not ATS optimized
- Matching resumes with job descriptions takes time

The goal of this project is to automate the resume screening process using AI and NLP.

### Main Objective
To build an intelligent system that can:

1. Read resumes
2. Extract meaningful information
3. Compare resumes with job descriptions
4. Generate matching scores
5. Recommend relevant jobs
6. Simulate ATS filtering

In [1]:
import kagglehub

path = kagglehub.dataset_download("youssefkhalil/resumes-images-datasets")
print("Path to dataset files:", path)

Path to dataset files: C:\Users\vanda\.cache\kagglehub\datasets\youssefkhalil\resumes-images-datasets\versions\1


#### STEP 1 — Check Dataset Structure

In [2]:
path = r"C:\Users\vanda\.cache\kagglehub\datasets\youssefkhalil\resumes-images-datasets\versions\1\Resumes Datasets\Bing_images"
import os
for root, dirs, files in os.walk(path):

    print("ROOT:", root)
    print("DIRS:", dirs[:5])
    print("FILES:", files[:5])
    print("="*50)
    break

ROOT: C:\Users\vanda\.cache\kagglehub\datasets\youssefkhalil\resumes-images-datasets\versions\1\Resumes Datasets\Bing_images
DIRS: ['Accountant resumes', 'Advocate resumes', 'Agricultural resumes', 'Apparel resumes', 'Architects resumes']
FILES: []


#### STEP 2 — Explore All Subfolders

In [3]:
import os
import pandas as pd
import pdfplumber
import pytesseract

from PIL import Image
import warnings
from docx import Document

# -------------------------------------------------
# Tesseract path for WINDOWS
# -------------------------------------------------
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

# -------------------------------------------------
# Main dataset path
# -------------------------------------------------
main_path = (
    r"C:\Users\vanda\.cache\kagglehub\datasets"
    r"\youssefkhalil\resumes-images-datasets"
    r"\versions\1\Resumes Datasets"
)

# Main folders
subfolders = ['Bing_images', 'resume_database', 'Scrapped_Resumes']

# -------------------------------------------------
# FILE READER FUNCTION
# -------------------------------------------------

def read_file(file_path):

    text = ""

    # =================================================
    # IMAGE FILES
    # =================================================
    if file_path.lower().endswith(('.png', '.jpg', '.jpeg')):
        warnings.filterwarnings( "ignore", message="Palette images with Transparency expressed in bytes")

        img = Image.open(file_path).convert("RGBA").convert("L")

        # Resize for faster OCR
        img = img.resize((800, 800))

        text = pytesseract.image_to_string(
            img,
            config='--oem 3 --psm 6'
        )

    # =================================================
    # PDF FILES
    # =================================================
    elif file_path.lower().endswith('.pdf'):

        with pdfplumber.open(file_path) as pdf:

            for page in pdf.pages:

                extracted = page.extract_text()

                if extracted:
                    text += extracted + "\n"

    # =================================================
    # DOCX FILES
    # =================================================
    elif file_path.lower().endswith('.docx'):

        doc = Document(file_path)

        for para in doc.paragraphs:
            text += para.text + "\n"

    # =================================================
    # TXT FILES
    # =================================================
    elif file_path.lower().endswith('.txt'):

        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:

            text = f.read()

    return text


# =====================================================
# PROCESS DATASET
# =====================================================

for subfolder in subfolders:

    print(f"\n{'='*60}")
    print(f"Processing Folder: {subfolder}")
    print(f"{'='*60}")

    subfolder_path = os.path.join(main_path, subfolder)

    data = []

    total_files = 0
    success_count = 0
    error_count = 0

    # Walk through nested folders
    for root, dirs, files in os.walk(subfolder_path):

        if len(files) == 0:
            continue

        print(f"\nReading Folder: {root}")

        for file in files:

            total_files += 1

            file_path = os.path.join(root, file)

            try:

                text = read_file(file_path)

                # Save only meaningful text
                if text and len(text.strip()) > 50:

                    category = os.path.basename(root)

                    data.append({
                        "resume_text": text,
                        "category": category,
                        "subfolder": subfolder,
                        "file_name": file
                    })

                    success_count += 1

                # Progress update
                if total_files % 50 == 0:
                    print(f"Processed: {total_files}")

            except Exception as e:

                error_count += 1

                print(f"\nError reading:")
                print(file_path)
                print(e)

    # =====================================================
    # SAVE CSV
    # =====================================================

    if len(data) > 0:

        df = pd.DataFrame(data)

        csv_name = f"{subfolder}.csv"

        df.to_csv(csv_name, index=False)

        print(f"\nSaved: {csv_name}")
        print(f"Rows Saved: {len(df)}")

    else:

        print(f"\nNo valid data found in {subfolder}")

    # =====================================================
    # SUMMARY
    # =====================================================

    print("\nFolder Summary")
    print(f"Total Files: {total_files}")
    print(f"Successful Reads: {success_count}")
    print(f"Errors: {error_count}")

# =====================================================
# COMBINE ALL CSV FILES
# =====================================================

print(f"\n{'='*60}")
print("Combining CSV Files")
print(f"{'='*60}")

csv_files = [
    "Bing_images.csv",
    "resume_database.csv",
    "Scrapped_Resumes.csv"
]

dfs = []

for file in csv_files:

    if os.path.exists(file) and os.path.getsize(file) > 0:

        try:

            df = pd.read_csv(file)

            if not df.empty:

                dfs.append(df)

                print(f"Loaded: {file} | Rows: {len(df)}")

        except Exception as e:

            print(f"Error loading {file}")
            print(e)

# =====================================================
# FINAL DATASET
# =====================================================

if len(dfs) > 0:

    final_df = pd.concat(dfs, ignore_index=True)

    final_df.to_csv("final_resume_dataset.csv", index=False)

    print(f"\nFinal Dataset Shape: {final_df.shape}")

    print("\nSaved: final_resume_dataset.csv")

else:

    print("\nNo valid CSV files found.")


Processing Folder: Bing_images

Reading Folder: C:\Users\vanda\.cache\kagglehub\datasets\youssefkhalil\resumes-images-datasets\versions\1\Resumes Datasets\Bing_images\Accountant resumes
Processed: 50

Reading Folder: C:\Users\vanda\.cache\kagglehub\datasets\youssefkhalil\resumes-images-datasets\versions\1\Resumes Datasets\Bing_images\Advocate resumes
Processed: 100
Processed: 150

Reading Folder: C:\Users\vanda\.cache\kagglehub\datasets\youssefkhalil\resumes-images-datasets\versions\1\Resumes Datasets\Bing_images\Agricultural resumes
Processed: 200

Reading Folder: C:\Users\vanda\.cache\kagglehub\datasets\youssefkhalil\resumes-images-datasets\versions\1\Resumes Datasets\Bing_images\Apparel resumes
Processed: 250

Reading Folder: C:\Users\vanda\.cache\kagglehub\datasets\youssefkhalil\resumes-images-datasets\versions\1\Resumes Datasets\Bing_images\Architects resumes
Processed: 300

Reading Folder: C:\Users\vanda\.cache\kagglehub\datasets\youssefkhalil\resumes-images-datasets\versions\1\

#### STEP — DATA VALIDATION & CLEANING

##### 1. Check Dataset Shape and info

In [4]:
final_df.shape
final_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10035 entries, 0 to 10034
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   resume_text  10035 non-null  object
 1   category     10035 non-null  object
 2   subfolder    10035 non-null  object
 3   file_name    10035 non-null  object
dtypes: object(4)
memory usage: 313.7+ KB


##### 2. Checking Missing Values

In [5]:
df.isnull().sum()

resume_text    0
category       0
subfolder      0
file_name      0
dtype: int64

In [4]:
import pandas as pd
df = pd.read_csv("final_resume_dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10035 entries, 0 to 10034
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   resume_text  10035 non-null  object
 1   category     10035 non-null  object
 2   subfolder    10035 non-null  object
 3   file_name    10035 non-null  object
dtypes: object(4)
memory usage: 313.7+ KB


#### Check Duplicates

In [5]:
df.duplicated().sum()

np.int64(0)

#### Check Empty Resume Texts

In [6]:
df[
    df['resume_text'].str.strip() == ''
]

,resume_text,category,subfolder,file_name


#### Check Category Distribution

In [7]:
df['category'].value_counts()

category
Accountant                 372
Arts                       346
Advocate                   321
Testing                    312
Designer                   289
                          ... 
ETL Developer               37
BPO resumes                 28
React Developer resumes     18
Blockchain resumes           9
Building                     7
Name: count, Length: 99, dtype: int64

#### Check Resume Length

In [8]:
df['length'] = df['resume_text'].apply(len)

df['length'].describe()

count    10035.000000
mean      2576.809567
std       2337.133084
min         52.000000
25%       1187.000000
50%       2104.000000
75%       2983.000000
max      48323.000000
Name: length, dtype: float64

#### CREATE A COPY OF DATASET

In [9]:
df['cleaned_text'] = df['resume_text']

#### IMPORT REQUIRED LIBRARIES

In [10]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

#### DOWNLOAD NLTK DATA

In [11]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vanda\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vanda\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vanda\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\vanda\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

#### CREATE CLEANING FUNCTION

In [12]:
def preprocess_text(text):

    # =========================================
    # Lowercase
    # =========================================
    text = text.lower()

    # =========================================
    # Remove URLs
    # =========================================
    text = re.sub(r'http\\S+', ' ', text)

    # =========================================
    # Remove Emails
    # =========================================
    text = re.sub(r'\\S+@\\S+', ' ', text)

    # =========================================
    # Remove Numbers
    # =========================================
    text = re.sub(r'\\d+', ' ', text)

    # =========================================
    # Remove Special Characters
    # =========================================
    text = re.sub(r'[^a-zA-Z ]', ' ', text)

    # =========================================
    # Remove Extra Spaces
    # =========================================
    text = re.sub(r'\\s+', ' ', text).strip()

    # =========================================
    # Tokenization
    # =========================================
    tokens = word_tokenize(text)

    # =========================================
    # Stopword Removal
    # =========================================
    stop_words = set(stopwords.words('english'))

    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    # =========================================
    # Lemmatization
    # =========================================
    lemmatizer = WordNetLemmatizer()

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    # =========================================
    # Join back to sentence
    # =========================================
    text = " ".join(tokens)

    return text

#### APPLY PREPROCESSING

In [13]:
df['cleaned_text'] = df[
    'resume_text'
].apply(preprocess_text)

#### CHECK OUTPUT

In [14]:
df[
    ['resume_text', 'cleaned_text']
].head()

,resume_text,cleaned_text
0,EDUCATION\n© MBA - Executive Leadership © Bach...,education mba executive leadership bachelor sc...
1,HOWARD GERRARD\n\ncoo ie me\n\nees |= Verifyin...,howard gerrard coo ie ee verifying financial d...
2,SENIOR ACCOUNTANT\n= Qo ° in\ninfo@resumekrafi...,senior accountant qo info resumekraficom chica...
3,"Olivia Ogilvy, Accountant\n1515 Pacific Ave, L...",olivia ogilvy accountant pacific ave los angel...
4,"Brooklyn, NY\nStephen Greet, CPA (128) 336-785...",brooklyn ny stephen greet cpa stephen beamjobs...


#### SAVE PREPROCESSED DATASET

In [15]:
df.to_csv(
    "preprocessed_resume_dataset.csv",
    index=False
)